# Train Multimodal Deep Learning Models

### Purpose

**This notebook develops and evaluates two multimodal deep learning models for 5-year survival prediction: a concatenation MLP baseline and an attention fusion model. Together they form the ablation study comparing naive fusion against learned modality weighting.**

### Objectives
- Define shared MLP encoder architectures for RNA and clinical modalities.
- Train a concatenation fusion model (no attention) as the deep learning baseline.
- Train an attention fusion model that learns per-patient modality weights.
- Evaluate both models against logistic regression and XGBoost baselines.
- Prototype modeling and evaluation steps before implementation in the pipeline.

### Workflow

1. Load and inspect assembled datasets
   - Load preprocessed feature matrices and outcome labels produced by the data assembly step.
   - Confirm expected shapes and verify sample alignment between features and outcome labels.

**Subsequent exploratory steps use the training and validation data only.**

2. Define encoder architecture
   - Define RNA encoder: 25k → 256 → 128 (MLP with ReLU and dropout).
   - Define clinical encoder: 98 → 64 → 64 (shallower MLP with ReLU and dropout).
   - Encoders are shared across both fusion models.

3. Train concatenation fusion model
   - Concatenate RNA and clinical embeddings.
   - Pass through a classification head (MLP → sigmoid).
   - Train end-to-end on the binary survival label using BCE loss.

4. Evaluate concatenation model
   - Generate validation predictions.
   - Compute ROC-AUC and AP against all prior baselines.
   - Evaluate risk-tier separation.

5. Train attention fusion model
   - Pass RNA and clinical embeddings through a learned attention network.
   - Compute per-patient modality weights; take weighted combination of embeddings.
   - Pass fused embedding through classification head.
   - Train end-to-end on the binary survival label using BCE loss.

6. Evaluate attention fusion model
   - Generate validation predictions.
   - Compute ROC-AUC and AP against all prior baselines.
   - Evaluate risk-tier separation.

7. Compare all models
   - Assemble full ablation table across all models: Clinical LR, RNA LR, XGBoost, Concat MLP, Attention fusion.
   - Summarize what each step in the ablation adds.

**Now evaluate on test data.**

8. Evaluate final models on the test set
   - Generate test predictions for both deep learning models.
   - Compute final evaluation metrics.

9. Validate modeling outputs
   - Confirm prediction counts match split sizes.
   - Verify sample alignment across datasets.
   - Verify no data leakage between splits.

10. Test multimodal module

## 1. Load and inspect assembled datasets
   - Load preprocessed feature matrices and outcome labels produced by the data assembly step.
   - Confirm expected shapes and verify sample alignment between features and outcome labels.

In [ ]:
# imports
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

import matplotlib.pyplot as plt

In [ ]:
# load assembled feature matrices and targets for each split
assembled_dir = Path("../data/processed/assembled")

X_train_df = pd.read_parquet(assembled_dir / "train/X_concat.parquet")
X_val_df   = pd.read_parquet(assembled_dir / "val/X_concat.parquet")
X_test_df  = pd.read_parquet(assembled_dir / "test/X_concat.parquet")

y_train = pd.read_parquet(assembled_dir / "train/y.parquet")["y"]
y_val   = pd.read_parquet(assembled_dir / "val/y.parquet")["y"]
y_test  = pd.read_parquet(assembled_dir / "test/y.parquet")["y"]

for split, X, y in [("train", X_train_df, y_train), ("val", X_val_df, y_val), ("test", X_test_df, y_test)]:
    assert X.index.equals(y.index), f"{split}: index mismatch between X and y"
    print(f"{split}: n={len(y)}, n_events={y.sum()}, features={X.shape[1]}")

display(X_train_df.head(), X_train_df.tail())

## Subsequent exploratory steps use the training and validation data only.

## 2. Define encoder architecture
   - Define RNA encoder: 25k → 256 → 128 (MLP with ReLU and dropout).
   - Define clinical encoder: 98 → 64 → 64 (shallower MLP with ReLU and dropout).
   - Encoders are shared across both fusion models.

In [ ]:
# define shared RNA and clinical encoder architectures
import torch
import torch.nn as nn

X_clin_train_df = pd.read_parquet(assembled_dir / "train/X_clinical.parquet")
X_rna_train_df  = pd.read_parquet(assembled_dir / "train/X_rna.parquet")

n_clin_features = X_clin_train_df.shape[1]
n_rna_features  = X_rna_train_df.shape[1]
print(f"Clinical features: {n_clin_features}, RNA features: {n_rna_features}")


class RNAEncoder(nn.Module):
    """MLP encoder for RNA features: 25k → 256 → 128."""
    def __init__(self, input_dim: int, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 128),       nn.ReLU(), nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class ClinicalEncoder(nn.Module):
    """MLP encoder for clinical features: 98 → 64 → 64."""
    def __init__(self, input_dim: int, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 64),        nn.ReLU(), nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


rna_enc  = RNAEncoder(n_rna_features)
clin_enc = ClinicalEncoder(n_clin_features)

dummy_rna  = torch.zeros(4, n_rna_features)
dummy_clin = torch.zeros(4, n_clin_features)
print(f"RNA embedding shape:      {rna_enc(dummy_rna).shape}")
print(f"Clinical embedding shape: {clin_enc(dummy_clin).shape}")

## 3. Train concatenation fusion model
   - Concatenate RNA and clinical embeddings.
   - Pass through a classification head (MLP → sigmoid).
   - Train end-to-end on the binary survival label using BCE loss.

In [ ]:
# define and train concatenation fusion model
from torch.utils.data import DataLoader, TensorDataset

def to_tensors(X_rna_df, X_clin_df, y):
    return (
        torch.tensor(X_rna_df.values,  dtype=torch.float32),
        torch.tensor(X_clin_df.values, dtype=torch.float32),
        torch.tensor(y.values,         dtype=torch.float32),
    )

X_clin_val_df  = pd.read_parquet(assembled_dir / "val/X_clinical.parquet")
X_rna_val_df   = pd.read_parquet(assembled_dir / "val/X_rna.parquet")

rna_train, clin_train, y_train_t = to_tensors(X_rna_train_df, X_clin_train_df, y_train)
rna_val,   clin_val,   y_val_t   = to_tensors(X_rna_val_df,   X_clin_val_df,   y_val)

train_loader = DataLoader(
    TensorDataset(rna_train, clin_train, y_train_t),
    batch_size=32, shuffle=True,
)


class ConcatFusionModel(nn.Module):
    """Concatenate RNA and clinical embeddings, pass through classification head."""
    def __init__(self, rna_encoder, clin_encoder, dropout: float = 0.3):
        super().__init__()
        self.rna_encoder  = rna_encoder
        self.clin_encoder = clin_encoder
        fused_dim = 128 + 64
        self.head = nn.Sequential(
            nn.Linear(fused_dim, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1), nn.Sigmoid(),
        )

    def forward(self, x_rna, x_clin):
        emb = torch.cat([self.rna_encoder(x_rna), self.clin_encoder(x_clin)], dim=1)
        return self.head(emb).squeeze(1)


concat_model = ConcatFusionModel(RNAEncoder(n_rna_features), ClinicalEncoder(n_clin_features))
optimizer    = torch.optim.Adam(concat_model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion    = nn.BCELoss()

n_epochs = 50
for epoch in range(1, n_epochs + 1):
    concat_model.train()
    for rna_batch, clin_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = criterion(concat_model(rna_batch, clin_batch), y_batch)
        loss.backward()
        optimizer.step()

    if epoch % 10 == 0:
        concat_model.eval()
        with torch.no_grad():
            val_loss = criterion(concat_model(rna_val, clin_val), y_val_t).item()
        print(f"Epoch {epoch:3d} — val loss: {val_loss:.4f}")

## 4. Evaluate concatenation model
   - Generate validation predictions.
   - Compute ROC-AUC and AP against all prior baselines.
   - Evaluate risk-tier separation.

In [ ]:
# evaluate concatenation model against all prior baselines
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

concat_model.eval()
with torch.no_grad():
    y_val_pred_concat = concat_model(rna_val, clin_val).detach().cpu().numpy()

baseline_val_df = pd.read_parquet("../models/baselines/predictions_val.parquet")
xgb_val_df      = pd.read_parquet("../models/xgboost/predictions_val.parquet")

models = [
    ("Clinical LR",  baseline_val_df["y_pred_clin"].values),
    ("RNA LR",       baseline_val_df["y_pred_rna"].values),
    ("XGBoost",      xgb_val_df["y_pred_xgb"].values),
    ("Concat MLP",   y_val_pred_concat),
]

metrics_df = pd.DataFrame([
    {"model": name, "val_roc_auc": roc_auc_score(y_val, preds), "val_ap": average_precision_score(y_val, preds)}
    for name, preds in models
])
display(metrics_df)

fig, ax = plt.subplots(figsize=(6, 5))
for name, preds in models:
    fpr, tpr, _ = roc_curve(y_val, preds)
    auc = roc_auc_score(y_val, preds)
    ax.plot(fpr, tpr, label=f"{name} ({auc:.3f})")
ax.plot([0, 1], [0, 1], "k--")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — Validation")
ax.legend()
plt.tight_layout()
plt.show()

def risk_tier_summary(y_true, y_pred, model_name):
    thresholds = np.percentile(y_pred, [80, 20])
    tiers = np.where(y_pred >= thresholds[0], "high",
             np.where(y_pred <= thresholds[1], "low", "mid"))
    return (
        pd.DataFrame({"y": y_true.values, "tier": tiers})
        .groupby("tier")["y"]
        .agg(n="count", events="sum", event_rate="mean")
        .reindex(["high", "mid", "low"])
        .assign(model=model_name)
    )

display(risk_tier_summary(y_val, y_val_pred_concat, "Concat MLP"))

## 5. Train attention fusion model
   - Pass RNA and clinical embeddings through a learned attention network.
   - Compute per-patient modality weights; take weighted combination of embeddings.
   - Pass fused embedding through classification head.
   - Train end-to-end on the binary survival label using BCE loss.


In [ ]:
# define and train attention fusion model
class AttentionFusionModel(nn.Module):
    """
    Modality-level attention fusion: learns per-patient weights over RNA and clinical embeddings.
    Both embeddings are projected to the same dimension before attention is applied.
    """
    def __init__(self, rna_encoder, clin_encoder, embed_dim: int = 128, dropout: float = 0.3):
        super().__init__()
        self.rna_encoder  = rna_encoder
        self.clin_encoder = clin_encoder
        self.clin_proj    = nn.Linear(64, embed_dim)
        self.attention    = nn.Linear(embed_dim, 1)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1), nn.Sigmoid(),
        )

    def forward(self, x_rna: torch.Tensor, x_clin: torch.Tensor) -> torch.Tensor:
        rna_emb  = self.rna_encoder(x_rna)
        clin_emb = self.clin_proj(self.clin_encoder(x_clin))

        stacked = torch.stack([rna_emb, clin_emb], dim=1)         # (B, 2, 128)
        weights = torch.softmax(self.attention(stacked), dim=1)    # (B, 2, 1)
        fused   = (stacked * weights).sum(dim=1)                   # (B, 128)

        return self.head(fused).squeeze(1)


attn_model = AttentionFusionModel(RNAEncoder(n_rna_features), ClinicalEncoder(n_clin_features))
optimizer  = torch.optim.Adam(attn_model.parameters(), lr=1e-3, weight_decay=1e-4)

for epoch in range(1, n_epochs + 1):
    attn_model.train()
    for rna_batch, clin_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = criterion(attn_model(rna_batch, clin_batch), y_batch)
        loss.backward()
        optimizer.step()

    if epoch % 10 == 0:
        attn_model.eval()
        with torch.no_grad():
            val_loss = criterion(attn_model(rna_val, clin_val), y_val_t).item()
        print(f"Epoch {epoch:3d} — val loss: {val_loss:.4f}")


## 6. Evaluate attention fusion model
   - Generate validation predictions.
   - Compute ROC-AUC and AP against all prior baselines.
   - Evaluate risk-tier separation.


In [ ]:
# evaluate attention model against all prior baselines
attn_model.eval()
with torch.no_grad():
    y_val_pred_attn = attn_model(rna_val, clin_val).detach().cpu().numpy()

models = [
    ("Clinical LR",  baseline_val_df["y_pred_clin"].values),
    ("RNA LR",       baseline_val_df["y_pred_rna"].values),
    ("XGBoost",      xgb_val_df["y_pred_xgb"].values),
    ("Concat MLP",   y_val_pred_concat),
    ("Attention",    y_val_pred_attn),
]

metrics_df = pd.DataFrame([
    {"model": name, "val_roc_auc": roc_auc_score(y_val, preds), "val_ap": average_precision_score(y_val, preds)}
    for name, preds in models
])
display(metrics_df)

fig, ax = plt.subplots(figsize=(6, 5))
for name, preds in models:
    fpr, tpr, _ = roc_curve(y_val, preds)
    auc = roc_auc_score(y_val, preds)
    ax.plot(fpr, tpr, label=f"{name} ({auc:.3f})")
ax.plot([0, 1], [0, 1], "k--")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — Validation")
ax.legend()
plt.tight_layout()
plt.show()

display(risk_tier_summary(y_val, y_val_pred_attn, "Attention"))


## 7. Compare all models
   - Assemble full ablation table across all models: Clinical LR, RNA LR, XGBoost, Concat MLP, Attention fusion.
   - Summarize what each step in the ablation adds.


## Now evaluate on test data.

## 8. Evaluate final models on the test set
   - Generate test predictions for both deep learning models.
   - Compute final evaluation metrics.

In [ ]:
# evaluate both deep learning models on the test set
X_clin_test_df = pd.read_parquet(assembled_dir / "test/X_clinical.parquet")
X_rna_test_df  = pd.read_parquet(assembled_dir / "test/X_rna.parquet")
rna_test, clin_test, _ = to_tensors(X_rna_test_df, X_clin_test_df, y_test)

concat_model.eval()
attn_model.eval()
with torch.no_grad():
    y_test_pred_concat = concat_model(rna_test, clin_test).detach().cpu().numpy()
    y_test_pred_attn   = attn_model(rna_test, clin_test).detach().cpu().numpy()

baseline_test_df = pd.read_parquet("../models/baselines/predictions_test.parquet")
xgb_test_df      = pd.read_parquet("../models/xgboost/predictions_test.parquet")

test_models = [
    ("Clinical LR", baseline_test_df["y_pred_clin"].values),
    ("RNA LR",      baseline_test_df["y_pred_rna"].values),
    ("XGBoost",     xgb_test_df["y_pred_xgb"].values),
    ("Concat MLP",  y_test_pred_concat),
    ("Attention",   y_test_pred_attn),
]

test_metrics_df = pd.DataFrame([
    {"model": name, "test_roc_auc": roc_auc_score(y_test, preds), "test_ap": average_precision_score(y_test, preds)}
    for name, preds in test_models
])
display(test_metrics_df)

## 9. Validate modeling outputs
   - Confirm prediction counts match split sizes.
   - Verify sample alignment across datasets.
   - Verify no data leakage between splits.

In [11]:
# confirm prediction counts match split sizes
assert len(y_val_pred_concat)  == len(y_val),  "concat val size mismatch"
assert len(y_val_pred_attn)    == len(y_val),  "attn val size mismatch"
assert len(y_test_pred_concat) == len(y_test), "concat test size mismatch"
assert len(y_test_pred_attn)   == len(y_test), "attn test size mismatch"

# verify no sample overlap between splits
assert len(set(X_rna_train_df.index) & set(X_rna_val_df.index))  == 0, "train/val overlap"
assert len(set(X_rna_train_df.index) & set(X_rna_test_df.index)) == 0, "train/test overlap"
assert len(set(X_rna_val_df.index)   & set(X_rna_test_df.index)) == 0, "val/test overlap"

print("All validation checks passed.")

All validation checks passed.


## 10. Test multimodal module